# 📊 Notebook 04 — Storytelling Visual: DRE, Margens e Riscos

**Projeto:** Análise de Custos e Margem por Categoria — Olist E-Commerce  
**Autor:** Ariel Marquezin  
**Stack:** Matplotlib · Seaborn · pandas  

---

## 🎯 Objetivo

Transformar os dados financeiros calculados nos notebooks anteriores em **visualizações de impacto**,
contando uma história clara seguindo a metodologia STAR:

1. **Visão Geral** — DRE consolidada (waterfall de resultado)
2. **Pareto** — Quais categorias concentram 80% da receita?
3. **Quadrante** — Receita × Margem (scatter estratégico)
4. **Heatmap** — Correlação entre métricas financeiras
5. **Tendência** — Evolução mensal de receita e MoM
6. **Risco** — Margem × Cancelamento (matriz de risco)
7. **Boxplot** — Distribuição de ticket médio por categoria
8. **Dashboard final** — Painel com os KPIs do projeto

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pyarrow.parquet as pq

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor'  : '#0d1117',
    'axes.facecolor'    : '#161b22',
    'axes.edgecolor'    : '#30363d',
    'axes.labelcolor'   : '#c9d1d9',
    'axes.titlecolor'   : '#f0f6fc',
    'xtick.color'       : '#8b949e',
    'ytick.color'       : '#8b949e',
    'text.color'        : '#c9d1d9',
    'grid.color'        : '#21262d',
    'grid.linestyle'    : '--',
    'grid.alpha'        : 0.6,
    'font.family'       : 'monospace',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})
sns.set_theme(style="dark", rc=plt.rcParams)

# Paleta de cores
C_CYAN    = '#00e5ff'
C_GREEN   = '#10b981'
C_YELLOW  = '#f59e0b'
C_RED     = '#ef4444'
C_PURPLE  = '#7c3aed'
C_BLUE    = '#3b82f6'
C_GRAY    = '#30363d'

GOLD_PATH    = "../data/gold/"
FIGURES_PATH = "../figures/"
os.makedirs(FIGURES_PATH, exist_ok=True)

print(f"Matplotlib : {plt.matplotlib.__version__}")
print(f"Seaborn    : {sns.__version__}")
print("Estilo configurado ✅")

In [ ]:
# ── 2. Carregar dados Gold ────────────────────────────────────────────────────
def load(name):
    return pq.read_table(os.path.join(GOLD_PATH, f"{name}.parquet")).to_pandas()

dre      = load("dre_por_categoria")
pareto   = load("pareto_receita")
pareto_c = load("pareto_cpv")
monthly  = load("tendencia_mensal")
risk     = load("matriz_risco")
giro     = load("giro_estoque")
metrics  = load("metricas_pedidos")

print("Dados carregados:")
for name, df in [("dre",dre),("pareto",pareto),("monthly",monthly),("risk",risk)]:
    print(f"  {name:<12} {len(df):>5,} linhas")

In [ ]:
# ── 3. FIGURA 1: Waterfall — DRE Consolidada ─────────────────────────────────
# Mostra visualmente a cascata Receita → Deduções → CPV → Lucro → EBITDA.

totals = {
    'Receita\nBruta'       :  dre['receita_bruta'].sum(),
    '(-) Cancelamentos'    : -dre['deducoes_cancelamentos'].sum(),
    '(-) Impostos'         : -dre['impostos'].sum(),
    'Receita\nLíquida'     :  dre['receita_liquida'].sum(),
    '(-) CPV'              : -dre['cpv'].sum(),
    'Lucro\nBruto'         :  dre['lucro_bruto'].sum(),
    '(-) Custo\nOperac.'   : -dre['custo_operacional'].sum(),
    'EBITDA'               :  dre['ebitda'].sum(),
}

labels = list(totals.keys())
values = list(totals.values())
anchors = [0, values[0], values[0]+values[1], 0,
           values[3], values[3]+values[4], 0, 0]
# recompute proper running anchors for waterfall
running = 0
bottoms, bar_vals, colors = [], [], []
totals_idx = {0, 3, 5, 7}  # indices of "total" bars

for i, v in enumerate(values):
    if i in totals_idx:
        if i == 0:
            bottoms.append(0)
            bar_vals.append(v)
            colors.append(C_CYAN)
            running = v
        elif i == 3:
            bottoms.append(0)
            bar_vals.append(running)
            colors.append(C_BLUE)
        elif i == 5:
            bottoms.append(0)
            bar_vals.append(running)
            colors.append(C_GREEN if running > 0 else C_RED)
        elif i == 7:
            bottoms.append(0)
            bar_vals.append(running)
            colors.append(C_GREEN if running > 0 else C_RED)
    else:
        bottom = running if v < 0 else running
        bottoms.append(min(running, running + v))
        bar_vals.append(abs(v))
        colors.append(C_RED if v < 0 else C_GREEN)
        running += v

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

bars = ax.bar(labels, bar_vals, bottom=bottoms, color=colors,
              width=0.6, edgecolor='#0d1117', linewidth=1.5)

# Linha de conexão
for i in range(len(labels) - 1):
    top = bottoms[i] + bar_vals[i]
    ax.plot([i + 0.3, i + 0.7], [top, top], color='#30363d', linewidth=1)

# Anotações
for i, (b, v, c) in enumerate(zip(bottoms, bar_vals, colors)):
    val = values[i]
    txt = f"R$ {abs(val/1e6):.1f}M"
    y   = b + v + (b + v) * 0.01
    ax.text(i, y + abs(values[0]) * 0.02, txt,
            ha='center', va='bottom', fontsize=9,
            color='white', fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))
ax.set_title('DRE Consolidada — Olist E-Commerce', fontsize=15, fontweight='bold',
             color='white', pad=20)
ax.set_ylabel('Valor (R$ Milhões)', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

legend_elements = [
    mpatches.Patch(color=C_CYAN,  label='Receita Bruta'),
    mpatches.Patch(color=C_RED,   label='Deduções / Custos'),
    mpatches.Patch(color=C_GREEN, label='Resultado Positivo'),
    mpatches.Patch(color=C_BLUE,  label='Subtotal'),
]
ax.legend(handles=legend_elements, loc='upper right',
          facecolor='#161b22', edgecolor='#30363d', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '01_waterfall_dre.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 1 salva ✅")

In [ ]:
# ── 4. FIGURA 2: Gráfico de Pareto — Receita por Categoria ───────────────────

top_n   = min(20, len(pareto))
df_p    = pareto.head(top_n).copy()

fig, ax1 = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
ax1.set_facecolor('#161b22')

bar_colors = [C_CYAN if row['participacao_acumulada_pct'] <= 80 else C_GRAY
              for _, row in df_p.iterrows()]

bars = ax1.bar(range(top_n), df_p['receita_liquida'] / 1e6,
               color=bar_colors, edgecolor='#0d1117', linewidth=0.8)

ax2 = ax1.twinx()
ax2.plot(range(top_n), df_p['participacao_acumulada_pct'],
         color=C_YELLOW, linewidth=2.5, marker='o', markersize=4, label='% Acumulado')
ax2.axhline(80, color=C_RED, linestyle='--', linewidth=1.5, alpha=0.8, label='80%')
ax2.set_ylim(0, 110)
ax2.set_ylabel('% Acumulado', color=C_YELLOW, fontsize=10)
ax2.tick_params(colors=C_YELLOW)
ax2.set_facecolor('#161b22')

cats = [c[:18] + '…' if len(c) > 18 else c for c in df_p['category']]
ax1.set_xticks(range(top_n))
ax1.set_xticklabels(cats, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Receita Líquida (R$ Milhões)', fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:.1f}M'))
ax1.set_title(f'Princípio de Pareto — Top {top_n} Categorias por Receita', fontsize=14,
              fontweight='bold', color='white', pad=15)
ax1.grid(axis='y', alpha=0.3)
ax1.set_axisbelow(True)

legend_elems = [
    mpatches.Patch(color=C_CYAN, label='Concentram 80% da Receita'),
    mpatches.Patch(color=C_GRAY, label='Demais categorias'),
]
ax1.legend(handles=legend_elems, loc='upper right',
           facecolor='#161b22', edgecolor='#30363d', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '02_pareto_receita.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 2 salva ✅")

In [ ]:
# ── 5. FIGURA 3: Scatter Estratégico — Receita × Margem ──────────────────────
# O quadrante mais importante do projeto: onde focar e onde cortar.

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

x_med = dre['receita_liquida'].median()
y_med = dre['margem_bruta_media_pct'].median()

quad_colors = {
    '⭐ Alto Volume / Alta Margem'  : C_GREEN,
    '📦 Alto Volume / Baixa Margem' : C_YELLOW,
    '💎 Nicho Rentável'             : C_CYAN,
    '⚠️  Baixo Volume / Baixa Margem': C_RED,
}

for quad, color in quad_colors.items():
    mask = dre['quadrant'] == quad
    subset = dre[mask]
    ax.scatter(
        subset['receita_liquida'] / 1e6,
        subset['margem_bruta_media_pct'],
        s=subset['total_orders'] / 30,
        color=color, alpha=0.75, edgecolors='white',
        linewidths=0.5, label=quad, zorder=3
    )

# Labels nas top categorias
top_labels = dre.nlargest(8, 'receita_liquida')
for _, row in top_labels.iterrows():
    cat = row['category'][:16] + '…' if len(row['category']) > 16 else row['category']
    ax.annotate(cat,
                xy=(row['receita_liquida']/1e6, row['margem_bruta_media_pct']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=7.5, color='#c9d1d9', alpha=0.9)

ax.axvline(x_med / 1e6, color='#30363d', linestyle='--', linewidth=1.2, alpha=0.8)
ax.axhline(y_med,       color='#30363d', linestyle='--', linewidth=1.2, alpha=0.8)

# Rótulos dos quadrantes
for txt, xy in [
    ('ALTO VOLUME\nALTA MARGEM',  (0.97, 0.97)),
    ('ALTO VOLUME\nBAIXA MARGEM', (0.97, 0.03)),
    ('NICHO\nRENTÁVEL',           (0.03, 0.97)),
    ('BAIXO IMPACTO',             (0.03, 0.03)),
]:
    ax.text(*xy, txt, transform=ax.transAxes,
            fontsize=7.5, color='#30363d', va={'0.97':'top','0.03':'bottom'}[str(xy[1])],
            ha={'0.97':'right','0.03':'left'}[str(xy[0])], fontstyle='italic')

ax.set_xlabel('Receita Líquida (R$ Milhões)', fontsize=11)
ax.set_ylabel('Margem Bruta Média (%)', fontsize=11)
ax.set_title('Quadrante Estratégico — Receita × Margem por Categoria\n'
             '(tamanho dos pontos = volume de pedidos)',
             fontsize=13, fontweight='bold', color='white', pad=15)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5),
          facecolor='#161b22', edgecolor='#30363d', fontsize=9)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '03_quadrante_estrategico.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 3 salva ✅")

In [ ]:
# ── 6. FIGURA 4: Heatmap de Correlação ───────────────────────────────────────

corr_cols = [
    'receita_liquida', 'cpv', 'lucro_bruto', 'ebitda',
    'margem_bruta_media_pct', 'margem_ebitda_pct',
    'cancel_rate_pct', 'ticket_medio', 'frete_medio',
    'nps_medio', 'prazo_entrega_medio_dias'
]
corr_labels = [
    'Receita Liq.', 'CPV', 'Lucro Bruto', 'EBITDA',
    'Margem Bruta%', 'Margem EBITDA%',
    'Taxa Cancel.%', 'Ticket Médio', 'Frete Médio',
    'NPS Médio', 'Prazo Entrega'
]

corr_matrix = dre[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap=cmap,
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='#0d1117',
    xticklabels=corr_labels, yticklabels=corr_labels,
    ax=ax, annot_kws={'size': 8},
    cbar_kws={'shrink': 0.8}
)

ax.set_title('Heatmap de Correlação — Métricas Financeiras por Categoria',
             fontsize=13, fontweight='bold', color='white', pad=15)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '04_heatmap_correlacao.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 4 salva ✅")

In [ ]:
# ── 7. FIGURA 5: Tendência Mensal de Receita + MoM ───────────────────────────

df_m = monthly.dropna(subset=['mom_receita_pct']).copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                 gridspec_kw={'height_ratios': [2.5, 1]})
for a in [ax1, ax2]:
    a.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')

x = range(len(df_m))

# Área sob a curva
ax1.fill_between(x, df_m['receita_bruta'] / 1e6, alpha=0.15, color=C_CYAN)
ax1.plot(x, df_m['receita_bruta'] / 1e6, color=C_CYAN, linewidth=2.5,
         marker='o', markersize=5, label='Receita Bruta')
ax1.fill_between(x, df_m['custo_frete_total'] / 1e6, alpha=0.15, color=C_RED)
ax1.plot(x, df_m['custo_frete_total'] / 1e6, color=C_RED, linewidth=1.5,
         linestyle='--', marker='s', markersize=4, label='Custo de Frete')

ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v:.1f}M'))
ax1.set_ylabel('Valor (R$ Milhões)', fontsize=10)
ax1.set_title('Tendência Mensal de Receita — Olist', fontsize=13,
              fontweight='bold', color='white', pad=12)
ax1.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)
ax1.grid(alpha=0.25)

# MoM
mom_colors = [C_GREEN if v >= 0 else C_RED for v in df_m['mom_receita_pct']]
ax2.bar(x, df_m['mom_receita_pct'], color=mom_colors, alpha=0.85,
        edgecolor='#0d1117', linewidth=0.5)
ax2.axhline(0, color='#30363d', linewidth=1)
ax2.set_ylabel('MoM Receita (%)', fontsize=9)
ax2.grid(axis='y', alpha=0.25)

plt.xticks(x, df_m['mes'], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '05_tendencia_mensal.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 5 salva ✅")

In [ ]:
# ── 8. FIGURA 6: Matriz de Risco — Margem × Cancelamento ─────────────────────

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

status_palette = {
    '🚨 Alto Risco'   : C_RED,
    '⚠️  Atenção'     : C_YELLOW,
    '🟡 Monitorar'    : C_BLUE,
    '✅ Saudável'     : C_GREEN,
}

for status, color in status_palette.items():
    mask = risk['status_financeiro'] == status
    sub  = risk[mask]
    ax.scatter(
        sub['cancel_rate_pct'],
        sub['margem_bruta_media_pct'],
        s=sub['total_orders'] / 25,
        color=color, alpha=0.8,
        edgecolors='white', linewidths=0.4,
        label=status, zorder=3
    )

# Labels nos mais críticos
for _, row in risk.head(6).iterrows():
    cat = row['category'][:14] + '…' if len(row['category']) > 14 else row['category']
    ax.annotate(cat,
                xy=(row['cancel_rate_pct'], row['margem_bruta_media_pct']),
                xytext=(5, 3), textcoords='offset points',
                fontsize=7.5, color='#c9d1d9')

# Zonas de risco
ax.axvspan(ax.get_xlim()[0] if ax.get_xlim()[0] > 0 else 0, 3,
           alpha=0.04, color=C_GREEN)
ax.axvspan(3, ax.get_xlim()[1] if ax.get_xlim()[1] < 20 else 20,
           alpha=0.04, color=C_RED)

ax.set_xlabel('Taxa de Cancelamento (%)', fontsize=11)
ax.set_ylabel('Margem Bruta Média (%)', fontsize=11)
ax.set_title('Matriz de Risco Financeiro — Margem × Cancelamento\n'
             '(tamanho dos pontos = volume de pedidos)',
             fontsize=13, fontweight='bold', color='white', pad=15)
ax.legend(loc='upper right', facecolor='#161b22', edgecolor='#30363d', fontsize=9)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '06_matriz_risco.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 6 salva ✅")

In [ ]:
# ── 9. FIGURA 7: Boxplot — Distribuição de Ticket Médio ──────────────────────
# Seaborn boxplot nas top 12 categorias por receita.

top12_cats = dre.nlargest(12, 'receita_liquida')['category'].tolist()
df_box = metrics[metrics['category'].isin(top12_cats)].copy()
df_box['category_short'] = df_box['category'].apply(
    lambda c: c[:16] + '…' if len(c) > 16 else c
)

# Ordenar por mediana
order = (df_box.groupby('category_short')['price']
         .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

sns.boxplot(
    data=df_box, x='category_short', y='price',
    order=order, palette='Blues_r',
    width=0.55, fliersize=2, linewidth=1,
    boxprops=dict(alpha=0.8),
    ax=ax
)

ax.set_xlabel('Categoria', fontsize=11)
ax.set_ylabel('Preço do Item (R$)', fontsize=11)
ax.set_title('Distribuição de Ticket por Categoria — Top 12 por Receita',
             fontsize=13, fontweight='bold', color='white', pad=15)
ax.set_ylim(0, df_box['price'].quantile(0.97))
plt.xticks(rotation=35, ha='right', fontsize=8.5)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '07_boxplot_ticket.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figura 7 salva ✅")

In [ ]:
# ── 10. FIGURA 8: Dashboard Final — KPIs do Projeto ──────────────────────────
# Painel consolidado com os principais números — ideal para o README do GitHub.

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
gs  = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# KPI boxes
def kpi_box(ax, title, value, subtitle='', color=C_CYAN):
    ax.set_facecolor('#161b22')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.5, 0.75, value, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.40, title, ha='center', va='center',
            fontsize=10, color='#8b949e', transform=ax.transAxes)
    if subtitle:
        ax.text(0.5, 0.18, subtitle, ha='center', va='center',
                fontsize=8, color='#30363d', transform=ax.transAxes)
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(1.5)
        spine.set_visible(True)

# Calcular KPIs
receita_total  = dre['receita_liquida'].sum()
lucro_total    = dre['lucro_bruto'].sum()
ebitda_total   = dre['ebitda'].sum()
margem_media   = dre['margem_bruta_media_pct'].mean()
total_pedidos  = dre['total_orders'].sum()
cancel_rate    = (dre['cancelled_orders'].sum() / dre['total_orders'].sum()) * 100
top_cat        = dre.nlargest(1, 'receita_liquida')['category'].values[0]
n_categorias   = len(dre)
pareto_80_n    = len(pareto[pareto['participacao_acumulada_pct'] <= 80])

# Row 0 — KPIs principais
kpi_box(fig.add_subplot(gs[0,0]), 'Receita Líquida Total',
        f'R$ {receita_total/1e6:.1f}M', color=C_CYAN)
kpi_box(fig.add_subplot(gs[0,1]), 'Lucro Bruto Total',
        f'R$ {lucro_total/1e6:.1f}M', color=C_GREEN)
kpi_box(fig.add_subplot(gs[0,2]), 'EBITDA Estimado',
        f'R$ {ebitda_total/1e6:.1f}M',
        color=C_GREEN if ebitda_total > 0 else C_RED)
kpi_box(fig.add_subplot(gs[0,3]), 'Margem Bruta Média',
        f'{margem_media:.1f}%', color=C_YELLOW)

# Row 1 — KPIs secundários
kpi_box(fig.add_subplot(gs[1,0]), 'Total de Pedidos',
        f'{total_pedidos:,.0f}', color=C_BLUE)
kpi_box(fig.add_subplot(gs[1,1]), 'Taxa de Cancelamento',
        f'{cancel_rate:.1f}%',
        color=C_RED if cancel_rate > 3 else C_YELLOW)
kpi_box(fig.add_subplot(gs[1,2]), 'Categorias Analisadas',
        f'{n_categorias}', 'com ≥ 50 pedidos', color=C_CYAN)
kpi_box(fig.add_subplot(gs[1,3]), 'Pareto: cats. = 80% receita',
        f'{pareto_80_n}', f'de {n_categorias} categorias', color=C_PURPLE)

# Row 2 — Mini gráfico de barras top 8 EBITDA
ax_bar = fig.add_subplot(gs[2, :])
ax_bar.set_facecolor('#161b22')
top8 = dre.nlargest(8, 'ebitda')
cats = [c[:18]+'…' if len(c)>18 else c for c in top8['category']]
bar_colors2 = [C_GREEN if v > 0 else C_RED for v in top8['ebitda']]
ax_bar.bar(cats, top8['ebitda'] / 1e6, color=bar_colors2,
           edgecolor='#0d1117', linewidth=0.8)
ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v:.1f}M'))
ax_bar.set_title('Top 8 Categorias por EBITDA', fontsize=11,
                 fontweight='bold', color='white', pad=10)
ax_bar.grid(axis='y', alpha=0.25)
ax_bar.set_axisbelow(True)
plt.setp(ax_bar.get_xticklabels(), rotation=25, ha='right', fontsize=8.5)

# Título geral
fig.suptitle('📊 Olist Cost Analysis — Dashboard Executivo',
             fontsize=16, fontweight='bold', color='white', y=1.01)

plt.savefig(os.path.join(FIGURES_PATH, '08_dashboard_final.png'),
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("Figura 8 (Dashboard) salva ✅")

In [ ]:
# ── 11. Listagem de figuras geradas ──────────────────────────────────────────
import os
print("Figuras geradas em ../figures/:")
for f in sorted(os.listdir(FIGURES_PATH)):
    size_kb = os.path.getsize(os.path.join(FIGURES_PATH, f)) / 1024
    print(f"  {f:<45} {size_kb:6.0f} KB")
print("\nNota book 04 concluído ✅ → próximo: 05_modelo.ipynb")

## ✅ Resumo — Notebook 04

| Figura | Tipo | Insight Principal |
|--------|------|-------------------|
| 01 Waterfall DRE | Matplotlib | Cascata Receita → EBITDA |
| 02 Pareto Receita | Matplotlib + twin axis | X cats = 80% da receita |
| 03 Quadrante Estratégico | Scatter Matplotlib | Receita × Margem por categoria |
| 04 Heatmap Correlação | Seaborn | Relação entre métricas financeiras |
| 05 Tendência Mensal | Matplotlib área + barras MoM | Evolução e sazonalidade |
| 06 Matriz de Risco | Scatter Matplotlib | Margem × Cancelamento |
| 07 Boxplot Ticket | Seaborn | Distribuição de preço por categoria |
| 08 Dashboard Final | GridSpec Matplotlib | KPIs executivos consolidados |

**Próximo →** `05_modelo.ipynb` — Scikit-learn: predição de cancelamento de pedidos.